# HEX* Parameter-Grid Benchmark

This notebook makes `map_type` and `search_type` ordinary members of the parameter grid. One trial function runs both unidirectional and bidirectional searches, and a map cache ensures both modes use the same generated map.


In [1]:
%load_ext autoreload
%autoreload 2

from collections.abc import Mapping
from datetime import datetime
from itertools import product
from pathlib import Path
import random
import time
import pandas as pd
import os

import gc
import gzip
from tqdm.auto import tqdm

from hexgrid import HexCoord, HexGrid, VelocityState, build_obstacle_map, get_neighbors_at_radius
from hexstar import HStarProblem, BidHStarSearch, construct_full_solution, h_cost_travel_time_euclidian
from hexplot import plot_hex_grid_2


In [2]:
SAVE_FOLDER = Path("RESULTS_TEST/PARAMETER_GRID")
SAVE_FOLDER.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 500

result_batch = []
solution_cache = {}

batch_num = 1
completed_trials = 0

In [3]:
NUM_RUNS = 10
FIRST_RUN = 0
CENTER = HexCoord(0, 0)
SAFE_RADIUS = 7
HEX_SIZE = 1


map_type_config = {
    "random_n": {"obs_density_min": 0.10, "obs_density_max": 0.40},
    "clustered_n": {
        "obs_density_min": 0.10,
        "obs_density_max": 0.40,
        "min_cluster_size": 15,
        "max_cluster_size": 100,
        "exclude_radius": SAFE_RADIUS,
    },
}

In [4]:
search_param_grid = {
    "map_type": [
        "random_n",
        "clustered_n",
    ],

    "search_type": [
        "unidirectional",
        #"bidirectional",
    ],
    "heuristic": [
        ("hex", h_cost_travel_time),
        ("euclid", h_cost_travel_time_euclidian),
    ],
    "radius": [
        15,
        25,
        35,
        #50,
        #75,
        #100,
    ],

    "ay_window_ms": [
        3,
        7,
    ],

    "jps_horizon": [
        1,
        3,
        5,
        10,
        None,
    ],

    "n_step": [
        1,
        2,
        5,
        #8,
    ],

    "a_min": [2],
    "a_max": [1],

    "join_tolerance": [50],
    "state_key_mode": ["location"],
}

In [5]:
def parameter_sets(grid):
    keys = list(grid)
    for values in product(*(grid[key] for key in keys)):
        yield dict(zip(keys, values))


def random_rim_pair(center, radius, rng, inset=0):
    ring = list(get_neighbors_at_radius(center, radius - inset))
    start = rng.choice(ring)
    goal = HexCoord(2 * center.q - start.q, 2 * center.r - start.r)
    return start, goal


def flatten_dict(data, prefix=""):
    flat = {}
    for key, value in (data or {}).items():
        output_key = f"{prefix}_{key}" if prefix else str(key)
        if isinstance(value, Mapping):
            flat.update(flatten_dict(value, output_key))
        else:
            flat[output_key] = value
    return flat


def save_results(df, folder=SAVE_FOLDER, prefix="hexstar_benchmark"):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = folder / f"{prefix}_{timestamp}.xlsx"
    df.to_excel(output_path, index=False)
    print(f"Saved: {output_path}")
    return output_path


In [6]:
def format_cache_key(cache_key):
    (
        map_type,
        seed,
        radius,
        search_type,
        heuristic,
        a_min,
        a_max,
        ay_window_ms,
        jps_horizon,
        n_step,
        join_tolerance,
        state_key_mode,
    ) = cache_key

    jps = "N" if jps_horizon is None else jps_horizon

    return (
        f"{map_type}"
        f"_s{seed}"
        f"_r{radius}"
        f"_{'bi' if search_type == 'bidirectional' else 'uni'}"
        f"_amin{a_min}"
        f"_amax{a_max}"
        f"_ay{ay_window_ms}"
        f"_jps{jps}"
        f"_n{n_step}"
        f"_jt{join_tolerance}"
        f"_{state_key_mode}"
        f"_{heuristic}"
    )

In [7]:
def generate_map(*, map_type, radius, center, seed, safe_radius=3, hex_size=1):
    if map_type not in map_type_config:
        raise ValueError(f"Unsupported map_type: {map_type}")

    config = map_type_config[map_type]
    rng = random.Random(seed)
    obs_density = rng.uniform(config["obs_density_min"], config["obs_density_max"])
    start, goal = random_rim_pair(center, radius, rng, inset=safe_radius)
    n_obstacles = int(obs_density * 3 * radius * (radius + 1))

    if map_type == "random_n":
        obstacle_spec = ("random_n", {"n": n_obstacles})
    else:
        obstacle_spec = (
            "clustered_n",
            {
                "n": n_obstacles,
                "min_cluster_size": config["min_cluster_size"],
                "max_cluster_size": config["max_cluster_size"],
            },
        )

    obstacles = set(build_obstacle_map(
        center=center,
        radius=radius,
        seed=seed,
        exclude=[start, goal],
        types=[obstacle_spec],
    ).keys())

    for clear_radius in range(safe_radius):
        obstacles -= get_neighbors_at_radius(start, clear_radius)
        obstacles -= get_neighbors_at_radius(goal, clear_radius)

    obstacles |= get_neighbors_at_radius(center, radius)

    return {
        "map_type": map_type,
        "seed": seed,
        "radius": radius,
        "center": center,
        "start": start,
        "goal": goal,
        "obstacles": obstacles,
        "hex_size": hex_size,
        "obs_density": obs_density,
        "num_obstacles": len(obstacles),
    }


In [8]:
def run_trial(*, map_data, search_type, heuristic, a_min, a_max, ay_window_ms,
              jps_horizon, n_step, join_tolerance, state_key_mode,
              solution_cache=None):
    if search_type not in {"unidirectional", "bidirectional"}:
        raise ValueError(f"Unsupported search_type: {search_type}")

    bidirectional = search_type == "bidirectional"
    cache_key = (
        map_data["map_type"], map_data["seed"], map_data["radius"],
        search_type, heuristic_name, a_min, a_max, ay_window_ms, jps_horizon, n_step,
        join_tolerance, state_key_mode,
    )

    search = solution = full_solution = benchmarks = None
    started = time.perf_counter()

    base_result = {
        "cache_key": cache_key,
        "map_type": map_data["map_type"],
        "search_type": search_type,
        "bidirectional": bidirectional,
        "a_min": a_min,
        "a_max": a_max,
        "ay_window_ms": ay_window_ms,
        "jps_horizon": jps_horizon,
        "n_step": n_step,
        "join_tolerance": join_tolerance,
        "state_key_mode": state_key_mode,
        "seed": map_data["seed"],
        "radius": map_data["radius"],
        "obs_density": map_data["obs_density"],
        "num_obstacles": map_data["num_obstacles"],
        "start": map_data["start"],
        "goal": map_data["goal"],
        "solution_filename": format_cache_key(cache_key) + '.jpg',
        "heuristic": heuristic_name,
    }

    try:
        grid = HexGrid(map_data["hex_size"], map_data["obstacles"])
        problem = HStarProblem(
            grid, map_data["start"], map_data["goal"], a_max, a_min,
            collision_radius=0,
            jps_horizon=jps_horizon,
            ay_window_ms=ay_window_ms,
            n_step=n_step,
            start_v=VelocityState(0, None),
            #goal_v=VelocityState(0, None),
            heuristic=heuristic,
        )
        search = BidHStarSearch(
            problem=problem,
            bidirectional=bidirectional,
            join_tolerance=join_tolerance,
            state_key_mode=state_key_mode,
            enable_benchmarking=True,
            progress_every=None,
        )
        solution = search.search()
        benchmarks = search.get_benchmarks()
        if solution is not None:
            full_solution = construct_full_solution(solution)

        result = base_result | {
            "success": solution is not None,
            "elapsed_ms_outer": (time.perf_counter() - started) * 1000,
            "path_length_full": len(full_solution) if full_solution else None,
            "error": None,
        }
        result.update(flatten_dict(benchmarks, prefix="benchmark"))

    except Exception as exc:
        if search is not None:
            try:
                benchmarks = search.get_benchmarks()
            except Exception:
                benchmarks = None
        result = base_result | {
            "success": False,
            "elapsed_ms_outer": (time.perf_counter() - started) * 1000,
            "path_length_full": None,
            "error": repr(exc),
        }
        result.update(flatten_dict(benchmarks, prefix="benchmark"))

    if solution_cache is not None:
        solution_cache[cache_key] = {
            "search": search,
            "solution": solution,
            "solution_path": full_solution,
            "benchmarks": benchmarks,
            "map_data": map_data,
            "result": result,
        }
    return result


In [9]:
def write_batch_results(
    result_batch,
    batch_num,
):
    df = pd.DataFrame(result_batch)

    path = (
        SAVE_FOLDER
        / f"bench_batch_{batch_num:04d}.xlsx"
    )

    df.to_excel(
        path,
        index=False,
        engine="openpyxl",
    )

    print(
        f"Saved batch {batch_num}: "
        f"{len(df)} rows"
    )

In [10]:
def write_batch_solution_maps(
    solution_cache,
    batch_num,
):
    output_folder = (
        SAVE_FOLDER
        / "soln_maps"
        / f"batch_{batch_num:04d}"
    )

    output_folder.mkdir(
        parents=True,
        exist_ok=True,
    )

    for cache_key, cache_data in solution_cache.items():

        solution_path = cache_data["solution_path"]

        # if solution_path is None:
        #     continue

        map_data = cache_data["map_data"]
        result = cache_data["result"]

        save_path = (
            output_folder
            / result["solution_filename"]
        )

        plot_hex_grid_2(
            obstacles=map_data["obstacles"],
            start=map_data["start"],
            center=map_data["center"],
            radius=map_data["radius"] + 5,
            goal=map_data["goal"],
            figsize=(5, 5),
            show_coords=False,
            linewidth=0,
            path=solution_path,
            draw_axes=False,
            show_axis_legend=False,
            save_path=str(save_path),
        )

    print(
        f"Saved {len(solution_cache)} "
        f"solution maps in batch {batch_num}"
    )

In [11]:
from tqdm.auto import tqdm

all_results = []
map_cache = {}
solution_cache = {}

parameter_combinations = list(parameter_sets(search_param_grid))
expected_trials = NUM_RUNS * len(parameter_combinations)

print(f"Planned trials: {expected_trials:,}")

with tqdm(
    total=expected_trials,
    desc="Benchmark Trials",
    unit="trial",
) as pbar:

    for seed in range(FIRST_RUN, FIRST_RUN + NUM_RUNS):
        for params in parameter_combinations:

            map_key = (
                params["map_type"],
                seed,
                params["radius"],
            )

            if map_key not in map_cache:
                map_cache[map_key] = generate_map(
                    map_type=params["map_type"],
                    radius=params["radius"],
                    center=CENTER,
                    seed=seed,
                    safe_radius=SAFE_RADIUS,
                    hex_size=HEX_SIZE,
                )

            trial_params = dict(params)
            trial_params.pop("map_type")
            trial_params.pop("radius")

            result = run_trial(
                map_data=map_cache[map_key],
                solution_cache=solution_cache,
                **trial_params,
            )
            
            result_batch.append(result)
            completed_trials += 1
            if completed_trials % BATCH_SIZE == 0:
            
                write_batch_results(
                    result_batch,
                    batch_num,
                )
            
                write_batch_solution_maps(
                    solution_cache,
                    batch_num,
                )
            
                result_batch.clear()
                solution_cache.clear()
            
                gc.collect()
            
                batch_num += 1
            all_results.append(result)

            pbar.set_postfix(
                map=params["map_type"],
                search=params["search_type"],
                radius=params["radius"],
                success=result["success"],
            )

            # if not result["success"]:
            #     tqdm.write(
            #         f"FAIL | seed={seed} | "
            #         f"map={params['map_type']} | "
            #         f"search={params['search_type']} | "
            #         f"radius={params['radius']} | "
            #         f"error={result.get('error')}"
            #     )

            pbar.update(1)
            
if result_batch:

    write_batch_results(
        result_batch,
        batch_num,
    )

    write_batch_solution_maps(
        solution_cache,
        batch_num,
    )

    result_batch.clear()
    solution_cache.clear()

    gc.collect()


Planned trials: 1,800


Benchmark Trials:   0%|          | 0/1800 [00:00<?, ?trial/s]

KeyboardInterrupt: 